# 09 Vegetation-Climate Linkage Analysis

This notebook links MODIS vegetation indicators with observed ERA5-derived ETCCDI climate extremes by hydroclimatic zone.

Design choice for reviewer robustness:
- MODIS vegetation products are observed records.
- Therefore the direct vegetation-climate linkage uses observed ERA5-derived ETCCDI indices, not future CMIP scenario years.
- The direct overlap period is 2001-2014 for GPP/NPP and NDVI/EVI because the available ERA5 ETCCDI archive currently ends in 2014.
- MODIS-only vegetation trends are also computed for the full available period through 2025.

Outputs are saved for later Methods, Results, supplementary material, and reviewer response.


## Cell 1 - Setup and Paths


In [ ]:
from pathlib import Path
import math
import re
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl

warnings.filterwarnings('ignore')

ROOT = (Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve())
MODIS_ROOT = ROOT / 'output' / 'modis_veg'
ERA5_INDEX_DIR = ROOT / 'output' / 'etccdi_fast' / 'era5_indices'
ZONES_FILE = ROOT / 'output' / 'zones' / 'hydroclimatic_zones_SA.nc'

OUT_ROOT = ROOT / 'output' / 'vegetation_climate_linkage'
TABLE_DIR = OUT_ROOT / 'tables'
FIG_DIR = OUT_ROOT / 'figures'
RASTER_DIR = OUT_ROOT / 'rasters'
LOG_DIR = OUT_ROOT / 'logs'
for d in [TABLE_DIR, FIG_DIR, RASTER_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

VEG_PRODUCTS = {
    'ndvi_evi': {'folder': MODIS_ROOT / 'ndvi_evi', 'pattern': 'mod13a2_ndvi_evi_*_clipped.nc'},
    'gpp': {'folder': MODIS_ROOT / 'gpp', 'pattern': 'mod17a2h_gpp_*_clipped.nc'},
    'npp': {'folder': MODIS_ROOT / 'npp', 'pattern': 'mod17a3hgf_npp_*_clipped.nc'},
}

CLIMATE_INDICES = ['PRCPTOT', 'RX1day', 'RX5day', 'SDII', 'R10mm', 'R20mm', 'CDD', 'CWD', 'TXx', 'TXn', 'TNx', 'TNn', 'DTR', 'SU', 'TR', 'FD', 'ID']
KEY_CLIMATE_INDICES = ['PRCPTOT', 'RX1day', 'CDD', 'TXx', 'TNn']
VEG_TARGETS = ['NDVI', 'EVI', 'GPP', 'NPP']
ZONE_IDS = list(range(7))
ZONE_LABELS = {z: f'Z{z+1}' for z in ZONE_IDS}

INDEX_TITLES = {
    'PRCPTOT': 'Annual wet-day precipitation',
    'RX1day': 'Maximum 1-day precipitation',
    'CDD': 'Consecutive dry days',
    'TXx': 'Annual maximum of daily Tmax',
    'TNn': 'Annual minimum of daily Tmin',
}

print('MODIS root:', MODIS_ROOT)
print('ERA5 ETCCDI root:', ERA5_INDEX_DIR)
print('Output:', OUT_ROOT)


## Cell 2 - Helper Functions


In [ ]:
def standardise_xy(ds):
    rename = {}
    for cand in ['latitude', 'y']:
        if cand in ds.coords or cand in ds.dims:
            rename[cand] = 'lat'
    for cand in ['longitude', 'x']:
        if cand in ds.coords or cand in ds.dims:
            rename[cand] = 'lon'
    if rename:
        ds = ds.rename(rename)
    if 'lon' in ds.coords and float(ds.lon.max()) > 180:
        ds = ds.assign_coords(lon=((ds.lon + 180) % 360) - 180).sortby('lon')
    if 'lat' in ds.coords:
        ds = ds.sortby('lat')
    if 'lon' in ds.coords:
        ds = ds.sortby('lon')
    return ds


def first_data_var(ds):
    return list(ds.data_vars)[0] if ds.data_vars else None


def parse_year(path):
    m = re.search(r'(19|20)\d{2}', path.name)
    return int(m.group(0)) if m else None


def force_numeric_da(da):
    da = standardise_xy(da.squeeze(drop=True))
    if np.issubdtype(da.dtype, np.timedelta64):
        da = da / np.timedelta64(1, 'D')
    elif not np.issubdtype(da.dtype, np.number):
        da = da.astype('float64')
    keep = {c: da.coords[c] for c in ['lat', 'lon', 'time', 'year'] if c in da.coords}
    da = da.reset_coords(drop=True)
    if keep:
        da = da.assign_coords(keep)
    return da.astype('float32')


def scale_vegetation_da(da, target):
    da = force_numeric_da(da)
    attrs = dict(da.attrs)
    scale = attrs.get('scale_factor', None)
    add = attrs.get('add_offset', 0)
    try:
        if scale is not None and float(scale) not in [0, 1]:
            da = da * float(scale) + float(add)
    except Exception:
        pass
    # Heuristics for common Earth Engine MODIS exports when scale_factor attr is not preserved.
    sample = da.where(np.isfinite(da)).isel({d: slice(0, min(5, da.sizes[d])) for d in da.dims if da.sizes[d] > 5}, missing_dims='ignore')
    try:
        p99 = float(sample.quantile(0.99).values)
    except Exception:
        p99 = float(da.max(skipna=True).values)
    target_upper = target.upper()
    if target_upper in ['NDVI', 'EVI'] and p99 > 2:
        da = da / 10000.0
    if target_upper in ['GPP', 'NPP'] and p99 > 1000:
        da = da / 10000.0
    if target_upper in ['NDVI', 'EVI']:
        da = da.where((da >= -0.2) & (da <= 1.1))
    else:
        da = da.where((da >= 0) & (da < 1000))
    return da.astype('float32')


def annual_reduce_veg(da, target):
    if 'time' in da.dims:
        if target.upper() in ['GPP']:
            return da.sum('time', skipna=True)
        return da.mean('time', skipna=True)
    return da


def zone_means_from_da(da, zone_da, year, variable, source):
    z = zone_da.interp(lat=da.lat, lon=da.lon, method='nearest')
    rows = []
    for zid in ZONE_IDS:
        val = da.where(z == zid).mean(['lat', 'lon'], skipna=True)
        rows.append({'year': int(year), 'zone_id': zid, 'zone': ZONE_LABELS[zid], 'variable': variable, 'source': source, 'zone_mean': float(val.values)})
    return rows


def mann_kendall_test(y):
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y)]
    n = len(y)
    if n < 8:
        return {'n': n, 'z': np.nan, 'p': np.nan, 'tau': np.nan, 'trend': 'insufficient'}
    s = sum(np.sign(y[k+1:] - y[k]).sum() for k in range(n - 1))
    _, counts = np.unique(y, return_counts=True)
    var_s = (n * (n - 1) * (2 * n + 5) - np.sum(counts * (counts - 1) * (2 * counts + 5))) / 18.0
    z = 0 if var_s <= 0 else ((s - 1) / math.sqrt(var_s) if s > 0 else ((s + 1) / math.sqrt(var_s) if s < 0 else 0))
    p = math.erfc(abs(z) / math.sqrt(2.0))
    tau = s / (0.5 * n * (n - 1))
    trend = 'increasing' if p < 0.05 and z > 0 else ('decreasing' if p < 0.05 and z < 0 else 'not_significant')
    return {'n': n, 'z': float(z), 'p': float(p), 'tau': float(tau), 'trend': trend}


def sen_slope(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]; y = y[mask]
    if len(y) < 2:
        return np.nan
    slopes = []
    for i in range(len(y) - 1):
        dx = x[i+1:] - x[i]
        slopes.extend(((y[i+1:] - y[i]) / dx)[dx != 0])
    return float(np.median(slopes)) if slopes else np.nan


def benjamini_hochberg(pvals):
    pvals = np.asarray(pvals, dtype=float)
    out = np.full_like(pvals, np.nan, dtype=float)
    valid = np.isfinite(pvals)
    pv = pvals[valid]
    if len(pv) == 0:
        return out
    order = np.argsort(pv)
    ranked = pv[order]
    adj = ranked * len(pv) / np.arange(1, len(pv) + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    tmp = np.empty_like(adj); tmp[order] = np.clip(adj, 0, 1)
    out[valid] = tmp
    return out


def corr_pvalue_from_r(r, n):
    # Normal approximation via Fisher transform; robust enough for screening tables without scipy dependency.
    if n < 4 or not np.isfinite(r) or abs(r) >= 1:
        return np.nan
    z = 0.5 * np.log((1 + r) / (1 - r)) * math.sqrt(n - 3)
    return math.erfc(abs(z) / math.sqrt(2.0))


## Cell 3 - Input Inventory


In [ ]:
failures = []
if not MODIS_ROOT.exists():
    raise FileNotFoundError(MODIS_ROOT)
if not ERA5_INDEX_DIR.exists():
    raise FileNotFoundError(ERA5_INDEX_DIR)
if not ZONES_FILE.exists():
    raise FileNotFoundError(ZONES_FILE)

with xr.open_dataset(ZONES_FILE) as zds:
    zname = 'zone' if 'zone' in zds.data_vars else first_data_var(zds)
    zone_da = force_numeric_da(zds[zname]).load()

inventory = []
for product, cfg in VEG_PRODUCTS.items():
    for f in sorted(cfg['folder'].glob(cfg['pattern'])):
        inventory.append({'product': product, 'year': parse_year(f), 'path': str(f.relative_to(ROOT)), 'size_mb': round(f.stat().st_size / 1024 / 1024, 2)})
veg_inventory = pd.DataFrame(inventory)
veg_inventory.to_csv(TABLE_DIR / 'modis_vegetation_input_inventory.csv', index=False)

print(veg_inventory.groupby('product').agg(n_files=('path', 'count'), first_year=('year', 'min'), last_year=('year', 'max')))
print('Zone grid:', dict(zone_da.sizes))


## Cell 4 - MODIS Annual Zone Means


In [ ]:
veg_rows = []
veg_failures = []

for product, cfg in VEG_PRODUCTS.items():
    for f in sorted(cfg['folder'].glob(cfg['pattern'])):
        year = parse_year(f)
        if year is None:
            veg_failures.append(f'NO_YEAR|{f}')
            continue
        try:
            with xr.open_dataset(f) as ds:
                ds = standardise_xy(ds)
                if product == 'ndvi_evi':
                    candidates = [v for v in ds.data_vars if v.lower() in ['ndvi', 'evi']]
                    if not candidates:
                        candidates = list(ds.data_vars)[:2]
                    for var in candidates:
                        target = 'NDVI' if 'ndvi' in var.lower() else ('EVI' if 'evi' in var.lower() else var.upper())
                        da = annual_reduce_veg(scale_vegetation_da(ds[var], target), target)
                        veg_rows.extend(zone_means_from_da(da, zone_da, year, target, product))
                elif product == 'gpp':
                    var = 'Gpp' if 'Gpp' in ds.data_vars else ('gpp' if 'gpp' in ds.data_vars else first_data_var(ds))
                    da = annual_reduce_veg(scale_vegetation_da(ds[var], 'GPP'), 'GPP')
                    veg_rows.extend(zone_means_from_da(da, zone_da, year, 'GPP', product))
                elif product == 'npp':
                    var = 'Npp' if 'Npp' in ds.data_vars else ('npp' if 'npp' in ds.data_vars else first_data_var(ds))
                    da = annual_reduce_veg(scale_vegetation_da(ds[var], 'NPP'), 'NPP')
                    veg_rows.extend(zone_means_from_da(da, zone_da, year, 'NPP', product))
        except Exception as exc:
            veg_failures.append(f'VEG_FAIL|{product}|{f.name}|{type(exc).__name__}: {exc}')
            print('[FAIL]', veg_failures[-1])

veg_zone = pd.DataFrame(veg_rows)
veg_zone.to_csv(TABLE_DIR / 'modis_vegetation_zone_annual_means.csv', index=False)
(LOG_DIR / 'modis_vegetation_processing_failures.txt').write_text('\n'.join(veg_failures), encoding='utf-8')
print('Vegetation zone rows:', len(veg_zone), 'failures:', len(veg_failures))
veg_zone.head()


## Cell 5 - ERA5 ETCCDI Zone Means for Observed Climate Linkage


In [ ]:
clim_rows = []
clim_failures = []

for f in sorted(ERA5_INDEX_DIR.glob('ERA5_*_etccdi.nc')):
    year = parse_year(f)
    if year is None:
        continue
    try:
        with xr.open_dataset(f) as ds:
            ds = standardise_xy(ds)
            for idx in CLIMATE_INDICES:
                if idx not in ds.data_vars:
                    continue
                da = force_numeric_da(ds[idx])
                clim_rows.extend(zone_means_from_da(da, zone_da, year, idx, 'ERA5_ETCCDI'))
    except Exception as exc:
        clim_failures.append(f'CLIM_FAIL|{f.name}|{type(exc).__name__}: {exc}')
        print('[FAIL]', clim_failures[-1])

clim_zone = pd.DataFrame(clim_rows)
clim_zone.to_csv(TABLE_DIR / 'era5_etccdi_zone_annual_means_for_veg_linkage.csv', index=False)
(LOG_DIR / 'era5_etccdi_zone_processing_failures.txt').write_text('\n'.join(clim_failures), encoding='utf-8')
print('Climate zone rows:', len(clim_zone), 'failures:', len(clim_failures))
print('Climate years:', clim_zone.year.min(), clim_zone.year.max())
clim_zone.head()


## Cell 6 - Build Linkage Dataset


In [ ]:
veg_wide = veg_zone.pivot_table(index=['year', 'zone', 'zone_id'], columns='variable', values='zone_mean').reset_index()
clim_wide = clim_zone.pivot_table(index=['year', 'zone', 'zone_id'], columns='variable', values='zone_mean').reset_index()
linkage = veg_wide.merge(clim_wide, on=['year', 'zone', 'zone_id'], how='inner')
linkage.to_csv(TABLE_DIR / 'vegetation_climate_linkage_dataset_zone_year.csv', index=False)

coverage = linkage.groupby('zone').agg(first_year=('year', 'min'), last_year=('year', 'max'), n_years=('year', 'nunique')).reset_index()
coverage.to_csv(TABLE_DIR / 'vegetation_climate_linkage_coverage_by_zone.csv', index=False)
print('Linkage shape:', linkage.shape)
print(coverage)
linkage.head()


## Cell 7 - Correlation and Lagged Correlation Analysis


In [ ]:
corr_rows = []
for zone, zdf in linkage.groupby('zone'):
    zdf = zdf.sort_values('year')
    for veg in VEG_TARGETS:
        if veg not in zdf.columns:
            continue
        for clim in CLIMATE_INDICES:
            if clim not in zdf.columns:
                continue
            for lag in [0, 1]:
                temp = zdf[['year', veg, clim]].copy()
                if lag == 1:
                    temp[clim] = temp[clim].shift(1)
                temp = temp.dropna()
                n = len(temp)
                if n < 8:
                    continue
                pearson = temp[veg].corr(temp[clim], method='pearson')
                spearman = temp[veg].corr(temp[clim], method='spearman')
                corr_rows.append({
                    'zone': zone,
                    'vegetation_metric': veg,
                    'climate_index': clim,
                    'lag_years': lag,
                    'n_years': n,
                    'first_year': int(temp.year.min()),
                    'last_year': int(temp.year.max()),
                    'pearson_r': pearson,
                    'pearson_p_approx': corr_pvalue_from_r(pearson, n),
                    'spearman_r': spearman,
                    'spearman_p_approx': corr_pvalue_from_r(spearman, n),
                })

corr_df = pd.DataFrame(corr_rows)
if not corr_df.empty:
    corr_df['pearson_p_fdr'] = np.nan
    corr_df['spearman_p_fdr'] = np.nan
    for _, locs in corr_df.groupby(['vegetation_metric', 'lag_years']).groups.items():
        corr_df.loc[locs, 'pearson_p_fdr'] = benjamini_hochberg(corr_df.loc[locs, 'pearson_p_approx'].values)
        corr_df.loc[locs, 'spearman_p_fdr'] = benjamini_hochberg(corr_df.loc[locs, 'spearman_p_approx'].values)
    corr_df['significant_spearman_fdr_0_05'] = corr_df['spearman_p_fdr'] < 0.05

corr_df.to_csv(TABLE_DIR / 'vegetation_climate_correlations_by_zone.csv', index=False)
key_corr = corr_df[corr_df['climate_index'].isin(KEY_CLIMATE_INDICES)].copy()
key_corr.to_csv(TABLE_DIR / 'vegetation_key_etccdi_correlations_by_zone.csv', index=False)
print('Correlation rows:', len(corr_df))
key_corr.sort_values('spearman_p_fdr').head(10)


## Cell 8 - Vegetation Trend Analysis


In [ ]:
trend_rows = []
for (zone, veg), sub in veg_zone.groupby(['zone', 'variable']):
    sub = sub.sort_values('year')
    if len(sub) < 8:
        continue
    mk = mann_kendall_test(sub['zone_mean'].values)
    slope = sen_slope(sub['year'].values, sub['zone_mean'].values)
    trend_rows.append({
        'zone': zone,
        'vegetation_metric': veg,
        'start_year': int(sub.year.min()),
        'end_year': int(sub.year.max()),
        'n_years': mk['n'],
        'sen_slope_per_year': slope,
        'sen_slope_per_decade': slope * 10 if np.isfinite(slope) else np.nan,
        'mk_tau': mk['tau'],
        'mk_z': mk['z'],
        'mk_p': mk['p'],
        'mk_trend': mk['trend'],
    })
veg_trends = pd.DataFrame(trend_rows)
if not veg_trends.empty:
    veg_trends['mk_p_fdr'] = np.nan
    for _, locs in veg_trends.groupby('vegetation_metric').groups.items():
        veg_trends.loc[locs, 'mk_p_fdr'] = benjamini_hochberg(veg_trends.loc[locs, 'mk_p'].values)
    veg_trends['significant_fdr_0_05'] = veg_trends['mk_p_fdr'] < 0.05
veg_trends.to_csv(TABLE_DIR / 'modis_vegetation_zone_trends.csv', index=False)
print('Vegetation trend rows:', len(veg_trends))
veg_trends.head()


## Cell 9 - Publication Figures


In [ ]:
mpl.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10, 'axes.titlesize': 11, 'axes.titleweight': 'bold', 'axes.labelsize': 10, 'axes.labelweight': 'bold', 'xtick.labelsize': 9, 'ytick.labelsize': 9, 'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False})

# Main correlation heatmap using key ETCCDI indices and lag 0 Spearman correlations.
plot_corr = key_corr[key_corr['lag_years'] == 0].copy()
for veg in VEG_TARGETS:
    sub = plot_corr[plot_corr['vegetation_metric'] == veg]
    if sub.empty:
        continue
    mat = sub.pivot_table(index='zone', columns='climate_index', values='spearman_r').reindex(index=[f'Z{i+1}' for i in ZONE_IDS], columns=KEY_CLIMATE_INDICES)
    sig = sub.pivot_table(index='zone', columns='climate_index', values='significant_spearman_fdr_0_05').reindex(index=mat.index, columns=mat.columns).fillna(False)
    fig, ax = plt.subplots(figsize=(8.6, 4.8))
    im = ax.imshow(mat.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(mat.columns))); ax.set_xticklabels(mat.columns, rotation=35, ha='right', fontweight='bold')
    ax.set_yticks(range(len(mat.index))); ax.set_yticklabels(mat.index, fontweight='bold')
    ax.set_title(f'{veg} response to observed climate extremes by hydroclimatic zone', loc='left')
    for i, zone in enumerate(mat.index):
        for j, clim in enumerate(mat.columns):
            val = mat.loc[zone, clim]
            txt = '' if pd.isna(val) else f'{val:.2f}' + ('*' if bool(sig.loc[zone, clim]) else '')
            ax.text(j, i, txt, ha='center', va='center', fontsize=8.5, fontweight='bold')
    cb = fig.colorbar(im, ax=ax, shrink=0.85); cb.set_label('Spearman correlation', fontweight='bold')
    fig.tight_layout()
    out_png = FIG_DIR / f'vegetation_climate_correlation_heatmap_{veg}.png'
    out_pdf = FIG_DIR / f'vegetation_climate_correlation_heatmap_{veg}.pdf'
    fig.savefig(out_png, dpi=300, bbox_inches='tight'); fig.savefig(out_pdf, bbox_inches='tight')
    plt.show()
    print('[OK]', out_png.relative_to(ROOT))

# Vegetation trend heatmap.
if not veg_trends.empty:
    mat = veg_trends.pivot_table(index='zone', columns='vegetation_metric', values='sen_slope_per_decade').reindex(index=[f'Z{i+1}' for i in ZONE_IDS], columns=[v for v in VEG_TARGETS if v in veg_trends.vegetation_metric.unique()])
    sig = veg_trends.pivot_table(index='zone', columns='vegetation_metric', values='significant_fdr_0_05').reindex(index=mat.index, columns=mat.columns).fillna(False)
    vmax = np.nanpercentile(np.abs(mat.values), 98)
    vmax = 1.0 if not np.isfinite(vmax) or vmax == 0 else vmax
    fig, ax = plt.subplots(figsize=(7.2, 4.8))
    im = ax.imshow(mat.values, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
    ax.set_xticks(range(len(mat.columns))); ax.set_xticklabels(mat.columns, fontweight='bold')
    ax.set_yticks(range(len(mat.index))); ax.set_yticklabels(mat.index, fontweight='bold')
    ax.set_title('MODIS vegetation trends by hydroclimatic zone', loc='left')
    for i, zone in enumerate(mat.index):
        for j, veg in enumerate(mat.columns):
            val = mat.loc[zone, veg]
            txt = '' if pd.isna(val) else f'{val:.3f}' + ('*' if bool(sig.loc[zone, veg]) else '')
            ax.text(j, i, txt, ha='center', va='center', fontsize=8.5, fontweight='bold')
    cb = fig.colorbar(im, ax=ax, shrink=0.85); cb.set_label('Sen slope per decade', fontweight='bold')
    fig.tight_layout()
    out_png = FIG_DIR / 'modis_vegetation_zone_trend_heatmap.png'
    out_pdf = FIG_DIR / 'modis_vegetation_zone_trend_heatmap.pdf'
    fig.savefig(out_png, dpi=300, bbox_inches='tight'); fig.savefig(out_pdf, bbox_inches='tight')
    plt.show()
    print('[OK]', out_png.relative_to(ROOT))


## Cell 10 - Save Summary


In [ ]:
summary = f"""# Phase 5 Vegetation-Climate Linkage Summary

## Purpose

This phase links observed MODIS vegetation indicators with observed ERA5-derived annual ETCCDI climate extremes by hydroclimatic zone.

## Inputs

- MODIS vegetation root: `{MODIS_ROOT.relative_to(ROOT)}`
- ERA5 annual ETCCDI indices: `{ERA5_INDEX_DIR.relative_to(ROOT)}`
- Hydroclimatic zones: `{ZONES_FILE.relative_to(ROOT)}`

## Available Vegetation Products

- NDVI/EVI: MOD13A2 annual files, 2000-2025
- GPP: MOD17A2H annual files, 2001-2025
- NPP: MOD17A3HGF annual files, 2001-2025

## Linkage Period

The direct observed vegetation-climate linkage uses the overlapping MODIS and ERA5 ETCCDI years available in the current archive. Based on the current ERA5 ETCCDI files, this is mainly 2001-2014.

## Main Outputs

- `tables/modis_vegetation_zone_annual_means.csv`
- `tables/era5_etccdi_zone_annual_means_for_veg_linkage.csv`
- `tables/vegetation_climate_linkage_dataset_zone_year.csv`
- `tables/vegetation_climate_correlations_by_zone.csv`
- `tables/vegetation_key_etccdi_correlations_by_zone.csv`
- `tables/modis_vegetation_zone_trends.csv`
- publication figures in `figures/`
- processing logs in `logs/`

## Reviewer-Readiness Notes

The direct linkage avoids mixing observed MODIS records with future CMIP scenario years. Correlations are reported by hydroclimatic zone, with lag-0 and lag-1 tests, approximate p-values, and FDR-adjusted significance. MODIS vegetation trends are analyzed separately over the full available MODIS period through 2025.
"""
out = OUT_ROOT / 'PHASE_5_VEGETATION_CLIMATE_LINKAGE_SUMMARY.md'
out.write_text(summary, encoding='utf-8')
print(out)
print(summary)


## Cell 11 - Combined Manuscript Vegetation-Climate Linkage Figure

This final cell combines the four vegetation-climate correlation heatmaps into one manuscript-ready 2x2 panel figure. Existing separate figures remain available for supplementary use.


In [ ]:
# Combined 2x2 vegetation-climate linkage panel for main manuscript.
PANEL_VEG_TARGETS = [v for v in ['NDVI', 'EVI', 'GPP', 'NPP'] if v in key_corr['vegetation_metric'].unique()]
PANEL_CLIMATE_INDICES = [idx for idx in ['PRCPTOT', 'RX1day', 'CDD', 'TXx', 'TNn'] if idx in key_corr['climate_index'].unique()]

if not PANEL_VEG_TARGETS or not PANEL_CLIMATE_INDICES:
    raise ValueError('No vegetation-climate correlation data available for combined panel. Run correlation cells first.')

mpl.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.titleweight': 'bold',
    'axes.labelsize': 10,
    'axes.labelweight': 'bold',
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'figure.dpi': 130,
})

fig, axes = plt.subplots(2, 2, figsize=(13, 8.8), constrained_layout=True)
axes = axes.ravel()
panel_letters = list('ABCD')
last_im = None

for ax, veg, letter in zip(axes, PANEL_VEG_TARGETS, panel_letters):
    sub = key_corr[(key_corr['vegetation_metric'] == veg) & (key_corr['lag_years'] == 0)].copy()
    mat = (
        sub.pivot_table(index='zone', columns='climate_index', values='spearman_r')
        .reindex(index=[f'Z{i+1}' for i in ZONE_IDS], columns=PANEL_CLIMATE_INDICES)
    )
    sig = (
        sub.pivot_table(index='zone', columns='climate_index', values='significant_spearman_fdr_0_05')
        .reindex(index=mat.index, columns=mat.columns)
        .fillna(False)
    )
    last_im = ax.imshow(mat.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    ax.set_title(f'{letter}. {veg}', loc='left', pad=7)
    ax.set_xticks(range(len(mat.columns)))
    ax.set_xticklabels(mat.columns, rotation=35, ha='right', fontweight='bold')
    ax.set_yticks(range(len(mat.index)))
    ax.set_yticklabels(mat.index, fontweight='bold')
    ax.set_xlabel('Climate extreme index', fontweight='bold')
    ax.set_ylabel('Hydroclimatic zone', fontweight='bold')
    for i, zone in enumerate(mat.index):
        for j, clim in enumerate(mat.columns):
            val = mat.loc[zone, clim]
            txt = '' if pd.isna(val) else f'{val:.2f}' + ('*' if bool(sig.loc[zone, clim]) else '')
            ax.text(j, i, txt, ha='center', va='center', fontsize=8.2, fontweight='bold', color='black')

# Hide any unused axes if fewer than four products are available.
for ax in axes[len(PANEL_VEG_TARGETS):]:
    ax.axis('off')

if last_im is not None:
    cb = fig.colorbar(last_im, ax=axes[:len(PANEL_VEG_TARGETS)], shrink=0.86, location='right')
    cb.set_label('Spearman correlation', fontweight='bold')

fig.suptitle('Observed vegetation response to climate extremes across hydroclimatic zones', fontsize=15, fontweight='bold')
out_png = FIG_DIR / 'MANUSCRIPT_Figure_vegetation_climate_linkage_panel.png'
out_pdf = FIG_DIR / 'MANUSCRIPT_Figure_vegetation_climate_linkage_panel.pdf'
fig.savefig(out_png, dpi=300, bbox_inches='tight')
fig.savefig(out_pdf, bbox_inches='tight')
plt.show()

print('[OK]', out_png.relative_to(ROOT))
print('[OK]', out_pdf.relative_to(ROOT))
print('Asterisk indicates FDR-adjusted Spearman significance at p < 0.05.')
